In [ ]:
"""Day 16 / Session 1 - Lab Assignment 1: Feature Engineering by Hand + a Leakage-Free Pipeline."""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def standardize_fit(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)  # zero-variance columns must never divide by zero
    return mean, std


def standardize_transform(X, mean, std):
    return (np.asarray(X, dtype=float) - mean) / std  # uses the passed-in (train) statistics


def one_hot(labels, categories):
    index = {c: i for i, c in enumerate(categories)}
    out = np.zeros((len(labels), len(categories)))
    for r, lab in enumerate(labels):
        j = index.get(lab)
        if j is not None:  # an unknown label leaves an all-zero row
            out[r, j] = 1.0
    return out


def build_pipeline(numeric_cols, categorical_cols):
    pre = ColumnTransformer([
        ("num", StandardScaler(), list(numeric_cols)),
        ("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical_cols)),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])